# Tables for the dashboards

Two consumers:
- **The Streamlit app** reads small summary tables (`app_*`), one per chart. Each is the output of a query
  in `sql/`, so the app can show the exact SQL behind every chart. They are also saved to `app_data/` so
  the public demo can run them from a SQLite snapshot without a database.
- **Power BI** reads a small star schema of views in an `analytics` schema: `fact_orders`,
  `dim_customers`, and the cohort table. The same data is exported to `powerbi/data/` as CSV, in case
  connecting Power BI to PostgreSQL is not an option.

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SQL = ROOT / "sql"
APP_DATA = ROOT / "app_data"
POWERBI_DATA = ROOT / "powerbi" / "data"
APP_DATA.mkdir(exist_ok=True)
POWERBI_DATA.mkdir(parents=True, exist_ok=True)

load_dotenv(ROOT / ".env")
engine = create_engine(os.environ["DATABASE_URL"])


def run_sql(name):
    return pd.read_sql(text((SQL / f"{name}.sql").read_text()), engine)

In [2]:
APP_QUERIES = [
    "headline_kpis", "monthly_kpis", "top_categories",
    "cohort_retention", "time_to_return", "return_drivers",
    "review_by_delay", "wait_vs_promise", "late_by_state",
    "rfm_summary",
]

sizes = []
for name in APP_QUERIES:
    df = run_sql(name)
    df.to_sql(f"app_{name}", engine, if_exists="replace", index=False)
    df.to_csv(APP_DATA / f"app_{name}.csv", index=False)
    sizes.append({"table": f"app_{name}", "rows": len(df), "kb": round((APP_DATA / f"app_{name}.csv").stat().st_size / 1024, 1)})

pd.DataFrame(sizes)

,table,rows,kb
0,app_headline_kpis,1,0.2
1,app_monthly_kpis,20,1.2
2,app_top_categories,15,0.7
3,app_cohort_retention,207,5.5
4,app_time_to_return,1,0.1
5,app_return_drivers,39,1.7
6,app_review_by_delay,8,0.4
7,app_wait_vs_promise,10,0.3
8,app_late_by_state,21,0.5
9,app_rfm_summary,7,0.5


In [3]:
with engine.begin() as conn:
    conn.execute(text((SQL / "powerbi_views.sql").read_text()))

exports = {
    "fact_orders": "SELECT * FROM analytics.fact_orders",
    "dim_customers": "SELECT * FROM analytics.dim_customers",
    "cohort_retention": "SELECT * FROM app_cohort_retention",
}
for name, query in exports.items():
    df = pd.read_sql(query, engine)
    df.to_csv(POWERBI_DATA / f"{name}.csv", index=False)
    print(f"{name}: {len(df):,} rows, {(POWERBI_DATA / f'{name}.csv').stat().st_size / 1e6:.1f} MB")

fact_orders: 99,092 rows, 18.1 MB


dim_customers: 96,096 rows, 11.5 MB
cohort_retention: 207 rows, 0.0 MB


In [4]:
check = pd.read_sql("""
SELECT
    d.segment,
    COUNT(*) AS orders,
    ROUND(SUM(f.payment_value), 0) AS revenue,
    ROUND(AVG(f.review_score), 2) AS avg_review
FROM analytics.fact_orders f
LEFT JOIN analytics.dim_customers d USING (customer_unique_id)
GROUP BY d.segment
ORDER BY d.segment
""", engine)
assert check["segment"].notna().all()
assert check["orders"].sum() == pd.read_sql("SELECT COUNT(*) AS n FROM analytics.fact_orders", engine)["n"].iloc[0]
check

,segment,orders,revenue,avg_review
0,1. Loyal,2371,356873.0,4.29
1,"2. Repeat, going quiet",2060,307041.0,4.24
2,3. New big spender,14884,4489048.0,4.09
3,4. New,22217,1621627.0,4.22
4,5. Cooling off,18859,2858901.0,3.96
5,6. Lost big spender,14530,4404548.0,4.06
6,7. Lost,23098,1675482.0,4.16
7,8. No completed order,1073,230961.0,1.59


The star schema joins cleanly: every order in `fact_orders` finds its customer in `dim_customers` (no
blank segment), and the order counts add up to the fact table. Customers whose orders were all canceled
get their own "No completed order" segment.